In [1]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path
import json

data_file = '../03_data/'
output_file = '../04_output/'
os.makedirs(output_file, exist_ok=True)

# Generate the Dictionary of Variable Names

In [2]:
def parse_variable_name_dict(txt_path):
    result = {}
    
    encodings = ["utf-8", "gbk", "utf-8-sig"]
    lines = None
    
    for enc in encodings:
        try:
            with open(txt_path, "r", encoding=enc) as f:
                lines = f.readlines()
            break
        except UnicodeDecodeError:
            continue

    pattern = re.compile(r"^\s*([A-Za-z_][A-Za-z0-9_]*)\s*\[(.*?)\]")
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        match = pattern.match(line)
        if match:
            var_name = match.group(1).strip()
            chinese_name = match.group(2).strip()
            result[var_name] = chinese_name
    
    return result

def save_json(obj, path, indent=2):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=indent)

var_name = {}
for file in os.listdir(data_file + 'var_name/'):
    var_name = {**var_name , **parse_variable_name_dict(data_file + 'var_name/' + file)}
var_name['EndDate'] = '统计日期'
var_name_inv = {v: k for k, v in var_name.items()}
save_json(var_name, data_file + 'financial_statement/var_name.json')
save_json(var_name_inv, data_file + 'financial_statement/var_name_inv.json')

# Generate the Table of Corporate Financial Data

In [3]:
var_list = ['证券代码', '统计截止日期',
            '资产总计', '流动资产合计', '非流动资产合计', '流动负债合计', '非流动负债合计', '负债合计', '无形资产净额', '固定资产净额',
            '基本每股收益', '净利润', '营业总成本', '营业成本', '销售费用', '管理费用', '研发费用', '财务费用', '减：所得税费用',
            '购建固定资产、无形资产和其他长期资产支付的现金', '固定资产折旧、油气资产折耗、生产性生物资产折旧', '无形资产摊销', '现金的期末余额',
            '股票代码', '统计日期', '行业代码C', '行业名称C', '所属省份', '所属城市', '办公地经度', '办公地纬度', '注册地经度', '注册地纬度', '首次上市日期', 
]
var_list_new = ['code', 'date',
            'tolast', 'shtast', 'lngast', 'shtdebt', 'lngdebt', 'toldebt', 'intangast', 'fixedast',
            'eps', 'ni', 'tolopcost', 'opcost', 'salefee', 'mngfee', 'rdfee', 'finfee', 'tax',
            'capex', 'depreciation', 'amortization', 'cash',
            'code', 'date', 'ind', 'indname', 'province', 'city', 'officelng', 'officelat', 'registerlng', 'registerlat', 'listdate', 
]
var_list = list(map(lambda x: var_name_inv[x], var_list))
var_list_new = {var_list[i]: var_list_new[i] for i in range(len(var_list))}

In [4]:
def pretreat_data(loc):
    global var_list, var_list_new
    df = pd.read_parquet(loc)
    df = df[[var for var in df.columns if var in var_list]]
    df.columns = [var_list_new[c] for c in df.columns]
    return df

basic_info = pretreat_data(data_file + 'financial_statement/basic_info.parquet')
balance_sheet = pretreat_data(data_file + 'financial_statement/balance_sheet.parquet')
income_stat = pretreat_data(data_file + 'financial_statement/income_stat.parquet')
cash_flow_stat_di = pretreat_data(data_file + 'financial_statement/cash_flow_stat_di.parquet')
cash_flow_stat_in = pretreat_data(data_file + 'financial_statement/cash_flow_stat_in.parquet')

In [5]:
financial_stat = balance_sheet.merge(
    right=income_stat.merge(
        right=cash_flow_stat_di.merge(
            right=cash_flow_stat_in,
            on=['code', 'date'],
            how='inner',
        ),
        on=['code', 'date'],
        how='inner',
    ),
    on=['code', 'date'],
    how='inner',
)

In [6]:
os.makedirs(output_file + 'initial_tables/', exist_ok=True)
financial_stat.to_stata(output_file + 'initial_tables/financial_stat.dta', version=118)
basic_info.to_stata(output_file + 'initial_tables/basic_info.dta', version=118)